## Model Training

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [1]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import recall_score, accuracy_score, precision_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import warnings

#### Import the CSV Data as Pandas DataFrame

In [2]:
df = pd.read_csv('data/heart999.csv')

In [3]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [4]:

df['RestingBP'] = df['RestingBP'].replace(0, np.nan)
df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)

In [5]:
df['RestingBP'] = df.groupby('Sex')['RestingBP'].transform(lambda x: x.fillna(x.median()))
df['Cholesterol'] = df.groupby('Sex')['Cholesterol'].transform(lambda x: x.fillna(x.median()))

#### Preparing X and Y variables

In [6]:
#dropped HeartDisease as it is target variable
X = df.drop(columns=['HeartDisease'])


In [7]:
X

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,M,ATA,140.0,289.0,0,Normal,172,N,0.0,Up
1,49,F,NAP,160.0,180.0,0,Normal,156,N,1.0,Flat
2,37,M,ATA,130.0,283.0,0,ST,98,N,0.0,Up
3,48,F,ASY,138.0,214.0,0,Normal,108,Y,1.5,Flat
4,54,M,NAP,150.0,195.0,0,Normal,122,N,0.0,Up
...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110.0,264.0,0,Normal,132,N,1.2,Flat
914,68,M,ASY,144.0,193.0,1,Normal,141,N,3.4,Flat
915,57,M,ASY,130.0,131.0,0,Normal,115,Y,1.2,Flat
916,57,F,ATA,130.0,236.0,0,LVH,174,N,0.0,Flat


In [8]:
y = df['HeartDisease']

In [9]:
y

0      0
1      1
2      0
3      1
4      0
      ..
913    1
914    1
915    1
916    1
917    0
Name: HeartDisease, Length: 918, dtype: int64

In [10]:
X.shape

(918, 11)

In [11]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)
X_train.shape, X_test.shape

((734, 11), (184, 11))

In [12]:
# Create Column Transformer with 3 types of transformers
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = RobustScaler()
oh_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
         ("RobustScaler", numeric_transformer, num_features),        
    ]
)

/var/folders/vk/rlj4x5g95nq_dq2fhn9j1y4w0000gn/T/ipykernel_14997/3932409590.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include="object").columns


In [13]:
X.shape

(918, 11)

In [14]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [15]:
X_train.shape

(734, 20)

#### Create an Evaluate Function to give all metrics after model Training

In [16]:
def evaluate_model(true, predicted):
    recall = recall_score(true, predicted)
    precision = precision_score(true, predicted)
    accuracy = accuracy_score(true, predicted)
    return recall, precision, accuracy


In [17]:
models = {
    "Logistic Regression": LogisticRegression(),
    "K-Neighbors Classifier": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest Classifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(), 
    "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier(),
    "SVM Classifier": SVC()
}
model_list = []
recall_list =[]
accuracy_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_recall , model_train_precision, model_train_accuracy = evaluate_model(y_train, y_train_pred)

    model_test_recall , model_test_precision, model_test_accuracy = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Recall: {:.4f}".format(model_train_recall))
    print("- Precision: {:.4f}".format(model_train_precision))
    print("- Accuracy: {:.4f}".format(model_train_accuracy))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Recall: {:.4f}".format(model_test_recall))
    print("- Precision: {:.4f}".format(model_test_precision))
    print("- Accuracy: {:.4f}".format(model_test_accuracy))
    recall_list.append(model_test_recall)
    accuracy_list.append(model_test_accuracy)
    
    print('='*35)
    print('\n')

Logistic Regression
Model performance for Training set
- Recall: 0.8842
- Precision: 0.8651
- Accuracy: 0.8597
----------------------------------
Model performance for Test set
- Recall: 0.9118
- Precision: 0.8942
- Accuracy: 0.8913


K-Neighbors Classifier
Model performance for Training set
- Recall: 0.9212
- Precision: 0.8738
- Accuracy: 0.8828
----------------------------------
Model performance for Test set
- Recall: 0.9020
- Precision: 0.9109
- Accuracy: 0.8967


Decision Tree
Model performance for Training set
- Recall: 1.0000
- Precision: 1.0000
- Accuracy: 1.0000
----------------------------------
Model performance for Test set
- Recall: 0.7451
- Precision: 0.8000
- Accuracy: 0.7554


Random Forest Classifier
Model performance for Training set
- Recall: 1.0000
- Precision: 1.0000
- Accuracy: 1.0000
----------------------------------
Model performance for Test set
- Recall: 0.9118
- Precision: 0.8857
- Accuracy: 0.8859


XGBClassifier
Model performance for Training set
- Recall:

In [18]:
pd.DataFrame(list(zip(model_list, recall_list, accuracy_list)), columns=['Model Name', 'Recall', 'Accuracy']).sort_values(by=["Recall"],ascending=False)

,Model Name,Recall,Accuracy
0,Logistic Regression,0.911765,0.891304
3,Random Forest Classifier,0.911765,0.885870
5,CatBoosting Classifier,0.911765,0.896739
7,SVM Classifier,0.911765,0.875000
1,K-Neighbors Classifier,0.901961,0.896739
6,AdaBoost Classifier,0.852941,0.864130
4,XGBClassifier,0.833333,0.853261
2,Decision Tree,0.745098,0.755435
